In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from functools import reduce

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

SILVER_TABLES = [
    f"{SILVER_SCHEMA}.team_standard_stats",
    f"{SILVER_SCHEMA}.match_results_football_data",
    f"{SILVER_SCHEMA}.football_data_org_matches",
]

GOLD_TABLE = f"{GOLD_SCHEMA}.dim_season"


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

BUSINESS_KEY = [
    "season_key",
]

REQUIRED_SILVER_COLUMNS = [
    "season_start_year",
]

REQUIRED_GOLD_COLUMNS = [
    "season_key",
    "season_start_year",
    "season_end_year",
    "season_code",
    "season_name",
]


# ---------------------------------------------------------------------------
# Validate source tables
# ---------------------------------------------------------------------------

source_dfs = []

for table_name in SILVER_TABLES:

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

    silver_df = spark.table(table_name)

    validate_columns_exist(
        silver_df,
        REQUIRED_SILVER_COLUMNS,
    )

    source_metrics = validate_required_not_null(
        silver_df,
        REQUIRED_SILVER_COLUMNS,
        fail_on_null=True,
    )

    if source_metrics["row_count"] == 0:
        raise ValueError(
            f"Required Silver table is empty: {table_name}"
        )

    print(
        f"Source validated: {table_name}, "
        f"rows={source_metrics['row_count']}"
    )

    source_dfs.append(
        silver_df.select(
            F.col("season_start_year")
            .cast("int")
            .alias("season_start_year")
        )
    )


# ---------------------------------------------------------------------------
# Consolidate season identifiers
# ---------------------------------------------------------------------------

# Extract distinct seasons from all relevant Silver sources.
# This includes historical FBref seasons and current API fixtures.
#
# Season identity is based on the starting calendar year.
# Repeated seasons across sources must produce only one Gold row.

combined_seasons_df = reduce(
    lambda left, right: left.unionByName(right),
    source_dfs,
)

seasons_df = (
    combined_seasons_df
    .select("season_start_year")
    .distinct()
)


# ---------------------------------------------------------------------------
# Validate season identifiers
# ---------------------------------------------------------------------------

season_metrics = validate_required_not_null(
    seasons_df,
    ["season_start_year"],
    fail_on_null=True,
)

if season_metrics["row_count"] == 0:
    raise ValueError(
        "No valid seasons found in Silver sources. "
        "Refusing to build dim_season."
    )

# Prevent invalid or corrupted season years from entering Gold.

invalid_seasons_df = (
    seasons_df
    .filter(
        (F.col("season_start_year") < 1900)
        | (F.col("season_start_year") > 2100)
    )
)

if invalid_seasons_df.limit(1).count() > 0:
    display(invalid_seasons_df)

    raise ValueError(
        "Invalid season_start_year values detected."
    )

print(
    "Distinct Silver seasons validated: "
    f"{season_metrics['row_count']}"
)


# ---------------------------------------------------------------------------
# Build Gold season dimension
# ---------------------------------------------------------------------------

gold_df = (
    seasons_df
    .withColumn(
        "season_key",
        F.col("season_start_year"),
    )
    .withColumn(
        "season_end_year",
        F.col("season_start_year") + F.lit(1),
    )
    .withColumn(
        "season_code",
        F.format_string(
            "%02d%02d",
            F.pmod(F.col("season_start_year"), F.lit(100)),
            F.pmod(F.col("season_end_year"), F.lit(100)),
        ),
    )
    .withColumn(
        "season_name",
        F.concat(
            F.col("season_start_year").cast("string"),
            F.lit("/"),
            F.format_string(
                "%02d",
                F.pmod(F.col("season_end_year"), F.lit(100)),
            ),
        ),
    )
    .select(
        F.col("season_key").cast("int"),
        F.col("season_start_year").cast("int"),
        F.col("season_end_year").cast("int"),
        F.col("season_code"),
        F.col("season_name"),
    )
)


# ---------------------------------------------------------------------------
# Validate Gold season dimension
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

# Season keys must be unique.

duplicate_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    display(duplicate_keys)

    raise ValueError(
        "Duplicate season_key values found in dim_season."
    )

# A season must end in the year following its start year.

invalid_year_pairs = (
    gold_df
    .filter(
        F.col("season_end_year")
        != F.col("season_start_year") + F.lit(1)
    )
    .limit(1)
    .count()
)

if invalid_year_pairs > 0:
    raise ValueError(
        "Invalid season start/end year relationship."
    )

# Validate the season key convention.

invalid_season_keys = (
    gold_df
    .filter(
        F.col("season_key")
        != F.col("season_start_year")
    )
    .limit(1)
    .count()
)

if invalid_season_keys > 0:
    raise ValueError(
        "season_key must equal season_start_year."
    )

# Ensure the Gold dimension contains exactly the season
# identifiers discovered in the Silver sources.

source_season_keys = seasons_df.select(
    F.col("season_start_year").alias("season_key")
)

gold_season_keys = gold_df.select("season_key")

missing_seasons = (
    source_season_keys
    .join(
        gold_season_keys,
        on="season_key",
        how="left_anti",
    )
    .limit(1)
    .count()
)

unexpected_seasons = (
    gold_season_keys
    .join(
        source_season_keys,
        on="season_key",
        how="left_anti",
    )
    .limit(1)
    .count()
)

if missing_seasons > 0 or unexpected_seasons > 0:
    raise ValueError(
        "Gold season coverage does not match Silver sources."
    )

print(
    "Gold validation successful: "
    f"{gold_count} unique seasons."
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for this small dimension.
# Season identifiers are derived deterministically from Silver.
# Rerunning the notebook does not append duplicate seasons.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_metrics = validate_required_not_null(
    saved_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

saved_count = saved_metrics["row_count"]

if saved_count != gold_count:
    raise ValueError(
        "Post-write validation failed: "
        f"expected {gold_count} seasons, "
        f"found {saved_count}."
    )

saved_duplicate_keys = (
    saved_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: "
        "duplicate season_key values."
    )

saved_season_keys = {
    row["season_key"]
    for row in saved_df
    .select("season_key")
    .distinct()
    .collect()
}

expected_season_keys = {
    row["season_key"]
    for row in source_season_keys
    .distinct()
    .collect()
}

if saved_season_keys != expected_season_keys:
    raise ValueError(
        "Post-write validation failed: "
        "saved seasons do not match Silver source coverage."
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD SEASON DIMENSION LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Total seasons: {saved_count}")
print(f"Earliest season: {min(saved_season_keys)}")
print(f"Latest season: {max(saved_season_keys)}")
print(f"Duplicate season keys: {saved_duplicate_keys}")
print("=" * 55)

display(
    saved_df.orderBy("season_key")
)
